In [3]:
%load_ext autoreload
%autoreload 2
from aml_detection.dataset_loader import load_dataset, summarise

df = load_dataset(nrows=200_000)
summarise(df)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-08-15 10:35:28.989 | INFO     | aml_detection.dataset_loader:load_raw:69 - Reading /Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/data/raw/HI-Small_Trans.csv (first 200,000 rows)
2026-08-15 10:35:29.120 | SUCCESS  | aml_detection.dataset_loader:load_raw:84 - Read 200,000 rows x 11 columns
2026-08-15 10:35:29.120 | INFO     | aml_detection.dataset_loader:clean:149 - Cleaning transactions
2026-08-15 10:35:29.277 | SUCCESS  | aml_detection.dataset_loader:clean:161 - Cleaned 200,000 rows | laundering 0.0125% | self-loops 73.6%
                     Bank Name  Bank ID  ...  Entity ID                 Entity Name
0          Portugal Bank #4507   331579  ...  80062E240  Sole Proprietorship #50438
1              Canada Bank #27      210  ...  800C998A0          Corporation #33520
2                  UK Bank #33    21884  ...  800C47F50          Partnership #35397
3           Ge

KeyError: Ellipsis

In [4]:
import pandas as pd
from aml_detection.config import RAW_DATA_DIR

acc = pd.read_csv(RAW_DATA_DIR / "HI-Small_accounts.csv")
print(acc.shape)
print(acc.dtypes)
acc.head()

(518581, 5)
Bank Name         object
Bank ID            int64
Account Number    object
Entity ID         object
Entity Name       object
dtype: object


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [6]:
acc = acc.assign(
    account_key=lambda d: d["Bank ID"].astype(str) + "_" + d["Account Number"]
)

# CRITICAL: check uniqueness before merging, or you silently duplicate rows
print("rows:", len(acc), "| unique keys:", acc["account_key"].nunique())

acc = acc.assign(
    entity_type=lambda d: d["Entity Name"].str.split("#").str[0].str.strip(),
    bank_country=lambda d: d["Bank Name"].str.split("Bank").str[0].str.strip(),
)
print(acc.entity_type.value_counts())
print(acc.bank_country.value_counts().head(10))
merged = df.merge(acc, left_on="src_account", right_on="account_key", how="left")

for col in ["entity_type", "bank_country"]:
    r = merged.groupby(col)["is_laundering"].mean().sort_values(ascending=False)
    print(f"\n{col} | overall {merged.is_laundering.mean():.4f}")
    print(r.head(8))

# leakage probe on the numeric suffix
merged["entity_num"] = merged["Entity Name"].str.extract(r"#(\d+)").astype(float)
print("\nsuffix corr:", merged["entity_num"].corr(merged["is_laundering"]).round(4))

rows: 518581 | unique keys: 518581
entity_type
Partnership            189683
Corporation            172351
Sole Proprietorship    149048
Country                  6692
Individual                740
Direct                     67
Name: count, dtype: int64
bank_country
National       36077
Savings        35234
First          35227
Germany        32733
               28250
Switzerland    24949
China          23489
France         21416
India          19359
Israel         19248
Name: count, dtype: int64

entity_type | overall 0.0001
entity_type
Sole Proprietorship    0.000327
Partnership            0.000043
Corporation            0.000000
Country                0.000000
Direct                 0.000000
Individual             0.000000
Name: is_laundering, dtype: float64

bank_country | overall 0.0001
bank_country
Oasis Thrift          0.002008
                      0.000972
National              0.000048
Oasis Bancorp         0.000000
Oasis Savings         0.000000
Oasis Federal         0.00000

In [7]:
import pandas as pd
from aml_detection.config import RAW_DATA_DIR
from aml_detection.dataset_loader import load_dataset

# ---- 1. Load the FULL file (no nrows cap) -------------------------------
# The first 200K rows are not representative; laundering is spread across
# the whole 10-day span.
df = load_dataset()

print(f"rows              : {len(df):,}")
print(f"laundering count  : {df.is_laundering.sum():,}")
print(f"laundering rate   : {df.is_laundering.mean():.5%}")
print(f"span              : {df.timestamp.min()} -> {df.timestamp.max()}")

# where do the positives actually sit in the file?
print("\npositives by day:")
print(df.groupby(df.timestamp.dt.date)["is_laundering"].agg(["sum", "mean"]))


# ---- 2. Accounts file, parsed properly ----------------------------------
acc = pd.read_csv(RAW_DATA_DIR / "HI-Small_accounts.csv")

VALID_ENTITY = ["Corporation", "Partnership", "Sole Proprietorship", "Individual"]

acc = acc.assign(
    account_key=lambda d: d["Bank ID"].astype(str) + "_" + d["Account Number"],
    entity_type=lambda d: d["Entity Name"]
        .str.split("#").str[0].str.strip()
        .where(lambda s: s.isin(VALID_ENTITY), "Other"),
    # "Portugal Bank #4507" -> Portugal ; "National Bank of Harrisburg" -> Named/US
    bank_country=lambda d: d["Bank Name"]
        .str.extract(r"^(.+?)\s+Bank\s+#\d+$")[0]
        .fillna("Named/US"),
)

assert acc.account_key.is_unique, "duplicate account keys -- merge would fan out rows"

print("\nentity_type:\n", acc.entity_type.value_counts())
print("\nbank_country (top 10):\n", acc.bank_country.value_counts().head(10))


# ---- 3. Leakage check on the full data ----------------------------------
merged = df.merge(
    acc[["account_key", "entity_type", "bank_country", "Entity Name"]],
    left_on="src_account", right_on="account_key", how="left",
)

print(f"\nunmatched accounts: {merged.account_key.isna().mean():.2%}")

overall = merged.is_laundering.mean()
for col in ["entity_type", "bank_country"]:
    rates = merged.groupby(col, observed=True)["is_laundering"].agg(["mean", "size"])
    rates = rates[rates["size"] >= 500].sort_values("mean", ascending=False)
    print(f"\n{col} | overall {overall:.5%}")
    print((rates.assign(lift=lambda d: d["mean"] / overall)).head(8))

# generation-artifact probe: does the entity number predict the label?
suffix = merged["Entity Name"].str.extract(r"#(\d+)")[0].astype(float)
print(f"\nentity suffix corr with label: {suffix.corr(merged.is_laundering):.4f}")
print("(near zero = no leakage, safe to ignore the suffix)")


2026-08-15 10:48:15.804 | INFO     | aml_detection.dataset_loader:load_raw:69 - Reading /Users/ruturajdesai/Desktop/Data-Science/projects/end_to_end/aml_detection/data/raw/HI-Small_Trans.csv
2026-08-15 10:48:18.600 | SUCCESS  | aml_detection.dataset_loader:load_raw:84 - Read 5,078,345 rows x 11 columns
2026-08-15 10:48:18.600 | INFO     | aml_detection.dataset_loader:clean:149 - Cleaning transactions
2026-08-15 10:48:22.387 | SUCCESS  | aml_detection.dataset_loader:clean:161 - Cleaned 5,078,345 rows | laundering 0.1019% | self-loops 11.6%
rows              : 5,078,345
laundering count  : 5,177
laundering rate   : 0.10194%
span              : 2022-09-01 00:00:00 -> 2022-09-18 16:18:00

positives by day:
            sum      mean
timestamp                
2022-09-01  322  0.000289
2022-09-02  408  0.000541
2022-09-03  391  0.001885
2022-09-04  407  0.001962
2022-09-05  471  0.000976
2022-09-06  531  0.001101
2022-09-07  497  0.001030
2022-09-08  539  0.001116
2022-09-09  514  0.000785
20

In [8]:
per_day = df.groupby(df.timestamp.dt.date).agg(
    rows=("is_laundering", "size"),
    positives=("is_laundering", "sum"),
    rate=("is_laundering", "mean"),
)
print(per_day.to_string())

               rows  positives      rate
timestamp                               
2022-09-01  1114921        322  0.000289
2022-09-02   754449        408  0.000541
2022-09-03   207382        391  0.001885
2022-09-04   207430        407  0.001962
2022-09-05   482650        471  0.000976
2022-09-06   482089        531  0.001101
2022-09-07   482751        497  0.001030
2022-09-08   482773        539  0.001116
2022-09-09   654467        514  0.000785
2022-09-10   208325        442  0.002122
2022-09-11      396        232  0.585859
2022-09-12      281        170  0.604982
2022-09-13      184        106  0.576087
2022-09-14      121         70  0.578512
2022-09-15       46         28  0.608696
2022-09-16       46         26  0.565217
2022-09-17       23         15  0.652174
2022-09-18       11          8  0.727273


In [ ]:
df=load_dataset()
print(df.timestamp.max())
print(f"{df.is_laundering.mean():.4%}")
print(df.is_burn_in.mean())